In [0]:
from pyspark.sql import functions as F
import re
import unicodedata

spark.sql("CREATE CATALOG IF NOT EXISTS company_ro")
spark.sql("CREATE SCHEMA IF NOT EXISTS company_ro.silver")


def normalize_name(value: str) -> str:
    value = value or ""
    value = unicodedata.normalize("NFKD", value)
    value = value.encode("ascii", "ignore").decode("ascii")
    value = value.upper().strip()
    value = re.sub(r"[^A-Z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value)
    return value.strip("_")


def normalize_columns(df):
    used = {}

    for old_name in df.columns:
        new_name = normalize_name(old_name)

        if new_name in used:
            used[new_name] += 1
            new_name = f"{new_name}_{used[new_name]}"
        else:
            used[new_name] = 1

        if old_name != new_name:
            df = df.withColumnRenamed(old_name, new_name)

    return df


def first_existing(df, candidates):
    normalized_candidates = [normalize_name(c) for c in candidates]

    for candidate in normalized_candidates:
        if candidate in df.columns:
            return F.col(candidate)

    return F.lit(None)


def clean_text(col):
    return F.trim(col.cast("string"))


def clean_digits(col):
    return F.regexp_replace(col.cast("string"), r"[^0-9]", "")


def normalize_caen_4(col):
    digits = clean_digits(col)

    return (
        F.when(digits.isNull(), F.lit(None))
         .when(digits == "", F.lit(None))
         .when(F.length(digits) <= 4, F.lpad(digits, 4, "0"))
         .otherwise(digits)
    )


def normalize_caen_3(col):
    digits = clean_digits(col)

    return (
        F.when(digits.isNull(), F.lit(None))
         .when(digits == "", F.lit(None))
         .when(F.length(digits) <= 3, F.lpad(digits, 3, "0"))
         .otherwise(digits)
    )

In [0]:
n_caen_raw = normalize_columns(spark.table("company_ro.bronze.n_caen_raw"))

display(n_caen_raw.limit(50))

print("Columns:")
print(n_caen_raw.columns)

In [0]:
dim_caen_source = (
    n_caen_raw
    .select(
        normalize_caen_4(first_existing(n_caen_raw, [
            "CLASA",
            "COD_CAEN",
            "CAEN",
            "CLASA_CAEN",
            "COD"
        ])).alias("cod_caen"),

        normalize_caen_3(first_existing(n_caen_raw, [
            "GRUPA",
            "GRUPA_CAEN"
        ])).alias("grupa"),

        normalize_caen_4(first_existing(n_caen_raw, [
            "CLASA",
            "COD_CAEN",
            "CAEN",
            "CLASA_CAEN",
            "COD"
        ])).alias("clasa"),

        clean_text(first_existing(n_caen_raw, [
            "DENUMIRE",
            "DENUMIRE_CAEN",
            "DESCRIERE",
            "DESCRIERE_CAEN"
        ])).alias("denumire")
    )
)

display(dim_caen_source.limit(100))

display(
    dim_caen_source
    .withColumn("cod_len", F.length(F.col("cod_caen")))
    .withColumn("grupa_len", F.length(F.col("grupa")))
    .groupBy("cod_len", "grupa_len")
    .count()
    .orderBy("cod_len", "grupa_len")
)

In [0]:
dim_caen_clean = (
    dim_caen_source
    .filter(F.col("cod_caen").isNotNull())
    .filter(F.col("cod_caen") != "")
    .filter(F.length(F.col("cod_caen")) == 4)
    .filter(F.col("denumire").isNotNull())
    .filter(F.col("denumire") != "")
    .withColumn(
        "grupa",
        F.coalesce(
            F.col("grupa"),
            F.substring(F.col("cod_caen"), 1, 3)
        )
    )
    .withColumn("clasa", F.col("cod_caen"))
    .dropDuplicates(["cod_caen"])
)

display(dim_caen_clean.orderBy("cod_caen").limit(100))
print("Rows:", dim_caen_clean.count())

In [0]:
dim_caen = (
    dim_caen_clean
    .withColumn("caen_key", F.sha2(F.col("cod_caen"), 256))
    .select(
        "caen_key",
        "cod_caen",
        "denumire",
        "grupa",
        "clasa"
    )
    .orderBy("cod_caen")
)

display(dim_caen.limit(100))

print("Rows:", dim_caen.count())
print("Columns:", dim_caen.columns)

In [0]:
(
    dim_caen
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("company_ro.silver.dim_caen")
)

In [0]:
display(spark.sql("""
SELECT
  cod_caen,
  denumire,
  grupa,
  clasa
FROM company_ro.silver.dim_caen
ORDER BY cod_caen
LIMIT 100
"""))

In [0]:
display(spark.sql("""
SELECT
  cod_caen,
  denumire,
  grupa,
  clasa
FROM company_ro.silver.dim_caen
WHERE cod_caen IN ('4711', '6820', '6201', '6202', '4941', '4120', '0210', '0111')
ORDER BY cod_caen
"""))

In [0]:
display(spark.sql("""
SELECT
  cod_caen,
  denumire,
  grupa,
  clasa
FROM company_ro.silver.dim_caen
WHERE cod_caen LIKE '62%'
   OR cod_caen LIKE '63%'
ORDER BY cod_caen
"""))